In [1]:
# English comment: 1. Install everything first, let Kaggle make its mess
print("--- Installing everything (Stage 1) ---")
!pip install -q flashy encodec demucs hydra-core hydra-colorlog julius num2words datasets torchmetrics phonemizer transformers einops local_attention contrastive_learner rotary_embedding_torch
!pip install -q git+https://github.com/facebookresearch/audiocraft.git@c5157b5bf14bf83449c17ea1eeb66c19fb4bc7f0

# English comment: 2. NOW, force-overwrite the core with the exact matching versions
print("--- Force-fixing Torch & Torchaudio to match (Stage 2) ---")
!pip install --no-cache-dir --force-reinstall \
    torch==2.4.0+cu121 \
    torchaudio==2.4.0+cu121 \
    torchvision==0.19.0+cu121 \
    numpy==1.26.4 \
    --index-url https://download.pytorch.org/whl/cu121

# English comment: 3. Final system path setup
import os
os.environ["PHONEMIZER_ESPEAK_LIBRARY"] = "/usr/lib/x86_64-linux-gnu/libespeak-ng.so"
os.environ["PHONEMIZER_ESPEAK_PATH"] = "/usr/bin/espeak-ng"

print("\n--- Final Verification ---")
try:
    import torch
    import torchaudio
    print(f"✅ Torch: {torch.__version__}") # Must be 2.4.0+cu121
    # Test the binary link directly
    from torchaudio.lib import _torchaudio
    print("✅ Torchaudio binary linked successfully!")
    print("🚀 READY! Run the training cell now.")
except Exception as e:
    print(f"❌ Verification failed: {e}")

--- Installing everything (Stage 1) ---
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 45.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 5.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.

In [2]:
!git clone -b Daniella/VoiceCraft_kaggle/1 https://github.com/daniellasolo7/VoiceCraft.git

Cloning into 'VoiceCraft'...
remote: Enumerating objects: 6907, done.
remote: Counting objects: 100% (6568/6568), done.
remote: Compressing objects: 100% (6459/6459), done.
remote: Total 6907 (delta 153), reused 6480 (delta 106), pack-reused 339 (from 3)
Receiving objects: 100% (6907/6907), 19.14 MiB | 23.39 MiB/s, done.
Resolving deltas: 100% (307/307), done.


In [3]:
# !git clone https://github.com/jasonppy/VoiceCraft.git

In [4]:
!export PRETRAINED_MODEL="/kaggle/input/datasets/daniellasolo/voicecraft-pretrained-models/giga330M.pth"
!export DATASET_DIR="/kaggle/input/datasets/daniellasolo/voicecraft-hebrew-fleurs"
!export EXP_ROOT="/kaggle/working/exp"
!export WORLD_SIZE=2

In [5]:
%cd /kaggle/working/VoiceCraft/

/kaggle/working/VoiceCraft


In [6]:
# Final adjusted command for 330M model training on Kaggle
!torchrun --nnodes=1 --rdzv-backend=c10d --rdzv-endpoint=localhost:41977 --nproc_per_node=2 \
./main.py \
--dataset_dir "/kaggle/input/datasets/daniellasolo/voicecraft-hebrew-fleurs" \
--exp_dir "/kaggle/working/exp/hebrew_fleurs/run1" \
--dataset "gigaspeech" \
--manifest_name "manifest" \
--num_decoder_layers 8 \
--d_model 2048 \
--nhead 16 \
--text_vocab_size 100 \
--text_pad_token 100 \
--num_steps 10000 \
--lr 0.00001 \
--batch_size 2 \
--gradient_accumulation_steps 16 \
--n_codebooks 4 \
--audio_vocab_size 2048 \
--n_special 4 \
--eos 2051 \
--num_workers 2 \
--val_every_n_steps 100 \
--print_every_n_steps 50 \
--reduced_eog 1 \
--warmup_fraction 0.1 \
--early_stop_threshold 0.001 \
--max_mask_portion 0.5 \
--early_stop_step 200 \
--audio_max_length 16 \
--load_model_from /kaggle/input/datasets/daniellasolo/voicecraft-pretrained-models/giga330M.pth

W0304 11:36:48.174000 139575265064064 torch/distributed/run.py:779] 
W0304 11:36:48.174000 139575265064064 torch/distributed/run.py:779] *****************************************
W0304 11:36:48.174000 139575265064064 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0304 11:36:48.174000 139575265064064 torch/distributed/run.py:779] *****************************************
2026-03-04 11:36:53.878040: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-04 11:36:53.878138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1